<h3> <b>Project Based Virtual Intern: Data Scientist ID/X Partners x Rakamin Academy

Author: Arga Bathara

Date: 3 August 2025

<h2> <b>Credit Risk Prediction

### <b>Introduction

Acting as a Data Scientist at ID/X Partners, this project aims to assist a multifinance client in enhancing their credit risk assessment and management. The main objective is to develop a machine learning model capable of predicting credit risk based on a provided dataset of historical loans, which includes both approved and rejected applications. An accurate predictive model will help the client optimize business decisions and minimize potential financial losses.

The model development process will follow these core steps:
* Data Understanding
* Exploratory Data Analysis (EDA)
* Data Preparation
* Data Modeling
* Evaluation

---

### <b>Preparing the Enviroment

In [125]:
import pandas as pd
import numpy as np

In [126]:
df = pd.read_csv('dataset/loan_data_2007_2014.csv', low_memory=False)

---

### <b>Data Understanding

In [127]:
df.head()

,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,0,1077501,1296599,5000,5000,4975.0,36 months,10.65,162.87,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1077430,1314167,2500,2500,2500.0,60 months,15.27,59.83,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1077175,1313524,2400,2400,2400.0,36 months,15.96,84.33,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1076863,1277178,10000,10000,10000.0,36 months,13.49,339.31,C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1075358,1311748,3000,3000,3000.0,60 months,12.69,67.79,B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [128]:
if df['id'].is_unique and df['member_id'].is_unique:
    print("The 'id' and 'member_id' columns are unique identifiers.")
else:
    print("The 'id' and 'member_id' columns are not unique identifiers.")

The 'id' and 'member_id' columns are unique identifiers.


In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Unnamed: 0                   466285 non-null  int64  
 1   id                           466285 non-null  int64  
 2   member_id                    466285 non-null  int64  
 3   loan_amnt                    466285 non-null  int64  
 4   funded_amnt                  466285 non-null  int64  
 5   funded_amnt_inv              466285 non-null  float64
 6   term                         466285 non-null  object 
 7   int_rate                     466285 non-null  float64
 8   installment                  466285 non-null  float64
 9   grade                        466285 non-null  object 
 10  sub_grade                    466285 non-null  object 
 11  emp_title                    438697 non-null  object 
 12  emp_length                   445277 non-null  object 
 13 

---

### <b>Feature Engineering

In [130]:
cols_to_drop = [
    
    # unique identifiers
    'Unnamed: 0',
    'id',
    'member_id',
    
    # free text
    'url',
    'desc',
    
    # all null / constant / others
    'zip_code',
    'annual_inc_joint',
    'dti_joint',
    'verification_status_joint',
    'open_acc_6m',
    'open_il_6m',
    'open_il_12m',
    'open_il_24m',
    'mths_since_rcnt_il',
    'total_bal_il',
    'il_util',
    'open_rv_12m',
    'open_rv_24m',
    'max_bal_bc',
    'all_util',
    'inq_fi',
    'total_cu_tl',
    'inq_last_12m',
    'mths_since_last_major_derog',
    'tot_coll_amt',
    'tot_cur_bal',
    'total_rev_hi_lim',
    
    # Expert judgment
    'sub_grade'
]

df.drop(columns=cols_to_drop, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 47 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   loan_amnt                   466285 non-null  int64  
 1   funded_amnt                 466285 non-null  int64  
 2   funded_amnt_inv             466285 non-null  float64
 3   term                        466285 non-null  object 
 4   int_rate                    466285 non-null  float64
 5   installment                 466285 non-null  float64
 6   grade                       466285 non-null  object 
 7   emp_title                   438697 non-null  object 
 8   emp_length                  445277 non-null  object 
 9   home_ownership              466285 non-null  object 
 10  annual_inc                  466281 non-null  float64
 11  verification_status         466285 non-null  object 
 12  issue_d                     466285 non-null  object 
 13  loan_status   

<b> Defining Target Variable

In [131]:
loan_status_percentages = df['loan_status'].value_counts(normalize=True) * 100
print(loan_status_percentages)

loan_status
Current                                                48.087757
Fully Paid                                             39.619332
Charged Off                                             9.109236
Late (31-120 days)                                      1.479782
In Grace Period                                         0.674695
Does not meet the credit policy. Status:Fully Paid      0.426349
Late (16-30 days)                                       0.261214
Default                                                 0.178432
Does not meet the credit policy. Status:Charged Off     0.163205
Name: proportion, dtype: float64


In [132]:
acc_now_deling_percentages = df['acc_now_delinq'].value_counts(normalize=True) * 100
print(acc_now_deling_percentages)

acc_now_delinq
0.0    99.628530
1.0     0.348092
2.0     0.019732
3.0     0.002359
4.0     0.000858
5.0     0.000429
Name: proportion, dtype: float64


In [133]:
pub_rec_percentages = df['pub_rec'].value_counts(normalize=True) * 100
print(pub_rec_percentages)

pub_rec
0.0     86.839204
1.0     11.378084
2.0      1.206204
3.0      0.345518
4.0      0.111527
5.0      0.059195
6.0      0.029169
7.0      0.013297
8.0      0.006220
9.0      0.003432
10.0     0.002788
11.0     0.001716
12.0     0.000429
18.0     0.000429
13.0     0.000429
40.0     0.000214
34.0     0.000214
21.0     0.000214
63.0     0.000214
54.0     0.000214
14.0     0.000214
15.0     0.000214
16.0     0.000214
19.0     0.000214
49.0     0.000214
17.0     0.000214
Name: proportion, dtype: float64


In [134]:
df['credit_risk'] = 'bad'

good_credit_criteria = (
    (df['loan_status'].isin([
        'Fully Paid', 
        'Current',
        'In Grace Period',
        'Does Not Meet the Credit Policy. Status: Fully Paid',
        ])) &
    (df['acc_now_delinq'] == 0) &
    (df['pub_rec'] == 0)
)

df.loc[good_credit_criteria, 'credit_risk'] = 'good'
df[['loan_status', 'acc_now_delinq', 'pub_rec', 'credit_risk']].head(10)

,loan_status,acc_now_delinq,pub_rec,credit_risk
0,Fully Paid,0.0,0.0,good
1,Charged Off,0.0,0.0,bad
2,Fully Paid,0.0,0.0,good
3,Fully Paid,0.0,0.0,good
4,Current,0.0,0.0,good
5,Fully Paid,0.0,0.0,good
6,Current,0.0,0.0,good
7,Fully Paid,0.0,0.0,good
8,Charged Off,0.0,0.0,bad
9,Charged Off,0.0,0.0,bad


<b> Data Cleaning

Column emp_length

In [135]:
df['emp_length'].unique()

array(['10+ years', '< 1 year', '1 year', '3 years', '8 years', '9 years',
       '4 years', '5 years', '6 years', '2 years', '7 years', nan],
      dtype=object)

In [136]:
df['emp_length_int'] = df['emp_length'].str.replace(r'[^0-9]+', '', regex=True).astype(float)
df['emp_length_int'].unique()

array([10.,  1.,  3.,  8.,  9.,  4.,  5.,  6.,  2.,  7., nan])

In [137]:
df.drop(columns=['emp_length'], inplace=True)

Column term_int

In [138]:
unique_terms = df['term'].unique()
print(unique_terms)

[' 36 months' ' 60 months']


In [139]:
df['term_int'] = df['term'].str.replace('months', '').astype(float)

unique_terms = df['term_int'].unique()
print(unique_terms)

[36. 60.]


In [140]:
df.drop(columns=['term'], inplace=True)

Column earliest_cr_line

In [141]:
df['earliest_cr_line'].head()

0    Jan-85
1    Apr-99
2    Nov-01
3    Feb-96
4    Jan-96
Name: earliest_cr_line, dtype: object

In [142]:
df['earliest_cr_line_date'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%y')
df['earliest_cr_line_date'].head()

0   1985-01-01
1   1999-04-01
2   2001-11-01
3   1996-02-01
4   1996-01-01
Name: earliest_cr_line_date, dtype: datetime64[ns]

---

### <b> Exploratory Data Analysis (EDA)

---